In [244]:
# control:
    # fuel cat 10 = 1, rest 0
    # set terrain to mean
    # from ignition, prev fire area 0
    # temp = 80 f
    # humidity = 20%


# varying:
    # u and v wind (-13, 13) increments of 1 (can try for -3,3)
    # humiditiy (0, 110) steps of 5
    # temp (277, 317) increments of 5 (convert from k to f for graph)

## Control Vector

In [245]:
import numpy as np
import pandas as pd

train_data = np.load("/Users/isaaclee/Wildfire_Research/data/recursive_train_data_norm.npy")

cols = [
    "fire_area_change_1hr_acres",      # Col 1
    "fire_area_change_2hr_acres",      # Col 2
    "fire_area_change_3hr_acres",      # Col 3
    "fire_area_current_norm",          # Col 4 (fire_area_curr / 19000)
    "fire_area_previous_norm",         # Col 5 (fire_area_prev / 14000)
    "avg_u10_norm",                    # Col 6 ((avg_u10 + 13.5) / 27)
    "avg_v10_norm",                    # Col 7 ((avg_v10 + 13.5) / 27)
    "avg_relative_humidity_norm",      # Col 8 (avg_rh / 100)
    "avg_temperature_norm",            # Col 9 ((avg_t - 260) / 60)
    "avg_terrain_gradient_x_norm",     # Col 10 ((grad_x + 0.14) / 0.28)
    "avg_terrain_gradient_y_norm",     # Col 11 ((grad_y + 0.14) / 0.28)
    "max_terrain_variation_norm",      # Col 12 (max_terr_var / 2500)
    "terrain_rms_roughness_norm",      # Col 13 (terr_rms / 425)
    "fuel_cat_1_fraction",             # Col 14
    "fuel_cat_2_fraction",             # Col 15
    "fuel_cat_3_fraction",             # Col 16
    "fuel_cat_4_fraction",             # Col 17
    "fuel_cat_5_fraction",             # Col 18
    "fuel_cat_6_fraction",             # Col 19
    "fuel_cat_7_fraction",             # Col 20
    "fuel_cat_8_fraction",             # Col 21
    "fuel_cat_9_fraction",             # Col 22
    "fuel_cat_10_fraction",            # Col 23
    "fuel_cat_11_fraction",            # Col 24
    "fuel_cat_12_fraction",            # Col 25
    "fuel_cat_13_fraction",            # Col 26
    "fuel_cat_14_fraction"             # Col 27
]


control = [0, 0, 0, 0, 0, 0.5, 0.5, 0.2, 0.664, 0.5, 0.5,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0]
#          1, 2, 3, 4, 5, 6,   7,   8,   9,     10,  11,  12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27
#                                               |____terrain____| |_____________________fuel cats______________________|

control = np.array(control)

## U and V Wind Sensitivity

In [246]:
print("max u: " + str((train_data[:, 5]*27-13.5).max()))
print("min u: " + str((train_data[:, 5]*27-13.5).min()))

print("max v: " + str((train_data[:, 6]*27-13.5).max()))
print("min v: " + str((train_data[:, 6]*27-13.5).min()))

start = -13
end = 13
inc = 1 # increments of 1

u_sens = np.tile(control, (int((end-start)/inc +1), 1))
v_sens = np.tile(control, (int((end-start)/inc +1), 1))

for i in range(0, int((end-start)/inc + 1)):
    u_sens[i, 5] = ((start + i*inc)+13.5)/27
    v_sens[i, 6] = ((start + i*inc)+13.5)/27

print(u_sens[:, 5])
print(v_sens[:, 6])


max u: 10.446324189930149
min u: -13.056928071798128
max v: 10.475102336542122
min v: -11.422725823578398
[0.01851852 0.05555556 0.09259259 0.12962963 0.16666667 0.2037037
 0.24074074 0.27777778 0.31481481 0.35185185 0.38888889 0.42592593
 0.46296296 0.5        0.53703704 0.57407407 0.61111111 0.64814815
 0.68518519 0.72222222 0.75925926 0.7962963  0.83333333 0.87037037
 0.90740741 0.94444444 0.98148148]
[0.01851852 0.05555556 0.09259259 0.12962963 0.16666667 0.2037037
 0.24074074 0.27777778 0.31481481 0.35185185 0.38888889 0.42592593
 0.46296296 0.5        0.53703704 0.57407407 0.61111111 0.64814815
 0.68518519 0.72222222 0.75925926 0.7962963  0.83333333 0.87037037
 0.90740741 0.94444444 0.98148148]


## Humidity Sensitivity

In [247]:
print("max humiditiy: " + str((train_data[:, 7]).max()))
print("min humiditiy: " + str((train_data[:, 7]).min()))

start = 0
end = 110
inc = 5

humidity_sens = np.tile(control, (int((end-start)/inc + 1), 1))

for i in range(0, int((end-start)/inc + 1)):
    humidity_sens[i, 7] = (start + i*inc)/100

print(humidity_sens[:, 7])

max humiditiy: 0.9972497509800543
min humiditiy: 0.041990353739537106
[0.   0.05 0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65
 0.7  0.75 0.8  0.85 0.9  0.95 1.   1.05 1.1 ]


## Temp Sensitivity

In [248]:
print("max temp: " + str((train_data[:, 8]).max()))
print("min temp: " + str((train_data[:, 8]).min()))

start = 277
end = 316
inc = 5

temp_sens = np.tile(control, (int((end-start)/inc + 1), 1))

for i in range(0, int((end-start)/inc + 1)):
    temp_sens[i, 8] = ((start + i*inc)-260)/60

print(temp_sens[:, 8])

max temp: 0.890898497482532
min temp: 0.09693690184706914
[0.28333333 0.36666667 0.45       0.53333333 0.61666667 0.7
 0.78333333 0.86666667]


In [249]:
print(u_sens.shape)
print(v_sens.shape)
print(humidity_sens.shape)
print(temp_sens.shape)

sens = np.concatenate((u_sens, v_sens, humidity_sens, temp_sens), axis=0)
print(sens.shape)

np.save('sensitivity_data.npy',sens)

(27, 27)
(27, 27)
(23, 27)
(8, 27)
(85, 27)
